# Interpret rotated MOFA feature loadings

The goal is to interpret the “significant” feature loadings for each factor in the rotated MOFA.

Please, skip the imports and read below

In [1]:
import os
import pathlib
import pandas as pd

## GSEA on rotated weights

In [2]:
feature_weights = pd.read_csv('02_mofa_rotations/feature_weights.csv', index_col=0)

In [3]:
feature_weights

,factor_1_loading,factor_2_loading,factor_3_loading,factor_4_loading,factor_5_loading,factor_6_loading,factor_7_loading,factor_8_loading,factor_9_loading,factor_10_loading
B cells_counts,-0.263414,0.135621,-0.063849,0.272134,0.092324,-0.473610,-0.026884,-0.079935,-0.094230,0.031354
CD4 T cells_counts,-0.253421,0.077913,-0.038406,0.137953,-0.008838,-0.328205,0.173752,-0.057801,-0.064513,-0.126420
CD8 T cells_counts,-0.250451,0.021434,-0.007835,0.126195,0.025626,-0.241555,0.175948,-0.095221,0.004853,-0.179677
Classical monocytes-1 CCR2_counts,0.009815,0.010104,0.137004,0.162554,0.383849,0.084139,0.026826,0.182148,-0.448065,0.045333
Classical monocytes-2 IL1B_counts,0.122378,-0.232056,0.115265,-0.135799,0.249824,0.191408,0.048765,0.259576,-0.007113,0.013736
...,...,...,...,...,...,...,...,...,...,...
gdT cells_MCM3AP,0.042782,-0.086327,-0.033649,-0.064328,0.032093,-0.087006,0.086528,-0.027901,-0.015420,0.022948
gdT cells_YBEY,-0.027121,0.003528,0.023548,-0.019335,-0.205659,0.013859,-0.126784,0.133876,-0.092683,0.114387
gdT cells_PCNT,0.068449,-0.010427,0.034594,0.039322,0.035352,0.009634,0.041969,0.006927,0.065717,0.049981
gdT cells_DIP2A,-0.013970,-0.152294,0.069326,-0.009583,-0.081685,-0.030688,-0.016093,0.013483,0.021806,0.042242


In [4]:
x = feature_weights.index.str.split('_')
x = x[x.str[1] != 'counts']
cell_types = x.str[0].unique().tolist()

In [5]:
BASE = pathlib.Path('./04_mofa_gsea')

In [6]:
%%time
for ct in cell_types:
    ct_dir = BASE / ct
    os.makedirs(ct_dir, exist_ok=True)
    for factor in feature_weights.columns:
        factor_info = pd.DataFrame(feature_weights[factor])
        factor_info.columns = ['loading']
        factor_info['cell_type'] = factor_info.index.str.split('_').str[0]
        factor_info['gene'] = factor_info.index.str.split('_').str[1]
        factor_info = factor_info[factor_info.gene != 'counts']
        # Previous version of GSEA:
        # factor_info['abs_loading'] = factor_info.loading.abs()
        # sort by absolute loading
        # factor_info = factor_info.sort_values(by='abs_loading', ascending=False)
        # for repeated genes take them from the fi
        # factor_info = factor_info.drop_duplicates(subset='gene', keep='first')
        # factor_info = factor_info[factor_info.cell_type == cell_type]

        # New version of GSEA:
        # Keep genes from other cell types in place, but garble their names
        factor_info.loc[factor_info.cell_type.ne(ct), 'gene'] += '_other'
        factor_info.set_index('gene').to_csv(ct_dir / f'{factor}.csv')

CPU times: user 17.1 s, sys: 231 ms, total: 17.3 s
Wall time: 17.4 s
